In [ ]:
import os
import time
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import matplotlib.pyplot as plt
from tqdm import tqdm

import timm  # pip install timm
from torch.amp import autocast, GradScaler


# ==========================================
# [증강] CutMix 유틸
# ==========================================
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2


# ==========================================
# [EMA] 버퍼(BN 통계) 복사 + Dynamic Decay
# ==========================================
class ModelEMA:
    def __init__(self, model, decay=0.9998):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        self.updates = 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        with torch.no_grad():
            for ema_p, p in zip(self.ema.parameters(), model.parameters()):
                ema_p.data.mul_(d).add_(p.data, alpha=1 - d)
            for ema_b, b in zip(self.ema.buffers(), model.buffers()):
                ema_b.data.copy_(b.data)


def get_parameter_groups(model, weight_decay=0.05):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim == 1 or name.endswith(".bias"):
            no_decay.append(param)
        else:
            decay.append(param)
    return [{'params': no_decay, 'weight_decay': 0.0},
            {'params': decay, 'weight_decay': weight_decay}]


# ==========================================
# [평가] clean train / val 공용 (TTA 옵션)
# ==========================================
@torch.no_grad()
def evaluate(model, loader, criterion, device, tta=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with autocast('cuda'):
            outputs = model(inputs)
            if tta:
                outputs = (outputs + model(torch.flip(inputs, dims=[3]))) / 2.0
            loss = criterion(outputs, labels)
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return total_loss / total, correct / total


# ==========================================
# [훈련]
# ==========================================
def train_model(model, train_loader, clean_train_loader, val_loader,
                criterion, optimizer, scheduler, scaler, device, epochs, save_path):
    best_acc = 0.0
    ema = ModelEMA(model, decay=0.9998)
    history = {'clean_train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

    print("\n[파인튜닝] ConvNeXt-Tiny (ImageNet-22k pretrained) + EMA/CutMix/warmup-cosine/TTA")
    print("사전학습 가중치 보호를 위해 LR을 낮춰 시작합니다.\n")

    for epoch in range(epochs):
        start_time = time.time()

        # ---------------- 훈련 ----------------
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1:03d}/{epochs:03d} [Train]")

        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)

            apply_cutmix = np.random.rand() < 0.5
            if apply_cutmix:
                lam = np.random.beta(1.0, 1.0)
                rand_index = torch.randperm(inputs.size(0), device=device)
                target_a, target_b = labels, labels[rand_index]
                bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
                inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
                lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (inputs.size(-1) * inputs.size(-2)))

            with autocast('cuda'):
                outputs = model(inputs)
                if apply_cutmix:
                    loss = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1. - lam)
                else:
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)

            running_loss += loss.item() * inputs.size(0)
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        epoch_train_loss = running_loss / len(train_loader.dataset)

        # ---- 진단: clean train acc (증강 X, EMA 모델) ----
        _, clean_train_acc = evaluate(ema.ema, clean_train_loader, criterion, device, tta=False)
        # ---- 검증 (EMA + TTA) ----
        epoch_val_loss, epoch_val_acc = evaluate(ema.ema, val_loader, criterion, device, tta=True)

        scheduler.step()

        history['clean_train_acc'].append(clean_train_acc)
        history['val_acc'].append(epoch_val_acc)
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)

        gap = clean_train_acc - epoch_val_acc
        elapsed = time.time() - start_time
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch + 1:03d} | {elapsed:.0f}s | LR {lr:.6f} | "
              f"CleanTrain {clean_train_acc:.4f} | Val {epoch_val_acc:.4f} | Gap {gap:+.4f}")

        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(ema.ema.state_dict(), save_path)
            print(f"  >> best 갱신 · 저장: {save_path}\n")
        else:
            print()

    print(f"\n훈련 종료. 최고 검증 정확도(EMA): {best_acc:.4f}")
    return history


def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['clean_train_acc'], label='Clean Train Acc')
    plt.plot(history['val_acc'], label='Val Acc (EMA+TTA)')
    plt.title('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.legend()
    graph_path = os.path.join(save_dir, "finetune_log.png")
    plt.savefig(graph_path)
    print(f"\n그래프 저장: {graph_path}")
    try:
        plt.show()
    except Exception:
        pass


# ==========================================
# [메인]
# ==========================================
if __name__ == "__main__":
    PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
    DATA_DIR = PATH_LOCAL if os.path.exists(PATH_LOCAL) else PATH_ONEDRIVE
    MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_convnext_model.pt")

    BATCH_SIZE = 64
    EPOCHS = 40           # 사전학습 파인튜닝은 수렴이 빨라 100까지 필요 없음
    LEARNING_RATE = 1e-4  # from-scratch 5e-4 → 파인튜닝은 낮춰야 사전학습 지식이 안 망가짐
    WARMUP_EPOCHS = 3
    NUM_WORKERS = 4       # 프리징 나면 0으로

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"학습 장치: {device}")

    # 파인튜닝은 증강을 살짝 약하게 (사전학습 표현을 크게 흔들지 않도록)
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.1),
    ])
    val_transform = transforms.Compose([
        transforms.Resize(236),           # crop_pct 0.95 근사 (224/0.95≈236)
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_root = os.path.join(DATA_DIR, 'train')
    val_root = os.path.join(DATA_DIR, 'val')

    print("데이터셋 로딩 중...")
    train_dataset = datasets.ImageFolder(root=train_root, transform=train_transform)
    val_dataset = datasets.ImageFolder(root=val_root, transform=val_transform)

    clean_full = datasets.ImageFolder(root=train_root, transform=val_transform)
    random.seed(42)
    clean_idx = random.sample(range(len(clean_full)), min(3000, len(clean_full)))
    clean_train_dataset = Subset(clean_full, clean_idx)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0), pin_memory=True)
    clean_train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0), pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0), pin_memory=True)

    NUM_CLASSES = len(train_dataset.classes)
    print(f"클래스 수: {NUM_CLASSES}")

    # ---- 백본 교체: ImageNet-22k 사전학습 ConvNeXt-Tiny ----
    print("ConvNeXt-Tiny(fb_in22k_ft_in1k) 사전학습 가중치 로딩...")
    model = timm.create_model(
        'convnext_tiny.fb_in22k_ft_in1k',
        pretrained=True,
        num_classes=NUM_CLASSES,
        drop_path_rate=0.1,       # DropPath는 timm 인자로 처리
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    param_groups = get_parameter_groups(model, weight_decay=0.05)
    optimizer = optim.AdamW(param_groups, lr=LEARNING_RATE)

    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS)
    cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

    scaler = GradScaler('cuda')

    history = train_model(model, train_loader, clean_train_loader, val_loader,
                          criterion, optimizer, scheduler, scaler, device, EPOCHS, MODEL_SAVE_PATH)
    plot_history(history, DATA_DIR)
